In [1]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd

pd.set_option("display.max_rows", 8)
!date

Wed Apr 30 12:46:18 PDT 2025


# Mean deaths and stillbirths averted by adding folate by wealth quintile


In [2]:
import vivarium_inputs
import db_queries
import gbd_mapping
import pathlib
from lsff_utils import config_utils

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [3]:
location = "india"
vehicle = "rice"

In [4]:
# Parameters
location = "ethiopia"
vehicle = "salt"


In [5]:
intervention_scenarios = config_utils.get_config()["custom_intervention_scenarios"].get(
    location, ["intervention"]
)
intervention_scenarios

['intervention_25_nrv', 'intervention_100_nrv']

## Forecasted births and stillbirths

In [6]:
asfr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.age_specific_fertility_rate,
    "estimate",
    location.title(),
    years=2022,
).value

In [7]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)

In [8]:
# Scale ASFR in each category down proportionally to the scale-down in TFR forecasted from GBD 2017
if location == "india":
    asfr_2030_to_2022_ratio = 1.61 / 1.91  # http://ihmeuw.org/6j8s
elif location == "nigeria":
    asfr_2030_to_2022_ratio = 4.43 / 4.96  # http://ihmeuw.org/6jqx
elif location == "ethiopia":
    asfr_2030_to_2022_ratio = 3.27 / 4.10  # http://ihmeuw.org/6j7d

asfr = asfr * asfr_2030_to_2022_ratio
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end
Ethiopia  Female  10.0       15.0     2022        2023        0.002040
                  15.0       20.0     2022        2023        0.036179
                  20.0       25.0     2022        2023        0.152703
                  25.0       30.0     2022        2023        0.146699
                                                                ...   
                  35.0       40.0     2022        2023        0.100878
                  40.0       45.0     2022        2023        0.052838
                  45.0       50.0     2022        2023        0.017421
                  50.0       55.0     2022        2023        0.001615
Name: value, Length: 9, dtype: float64

In [9]:
asfr = (
    asfr.reset_index()
    .assign(year_start=2030, year_end=2031)
    .set_index(asfr.index.names)
    .value
)
asfr.sort_values()

location  sex     age_start  age_end    year_start  year_end
Ethiopia  Female  0.000000   0.019178   2030        2031        0.000000
                  0.019178   0.076712   2030        2031        0.000000
                  0.076712   0.500000   2030        2031        0.000000
                  0.500000   1.000000   2030        2031        0.000000
                                                                  ...   
                  35.000000  40.000000  2030        2031        0.100878
                  30.000000  35.000000  2030        2031        0.142050
                  25.000000  30.000000  2030        2031        0.146699
                  20.000000  25.000000  2030        2031        0.152703
Name: value, Length: 50, dtype: float64

In [10]:
from vivarium_inputs import utilities
from vivarium_inputs.utility_data import get_location_id
from vivarium_gbd_access.gbd import get_age_group_id, SEX, RELEASE_IDS

In [11]:
def get_population_future(location, year):
    # Cobbled together from pieces of vivarium_inputs and vivarium_gbd_access
    # TODO: vivarium_inputs should be able to get forecasted pop!
    location_id = get_location_id(location)
    year_id = year
    data = db_queries.get_population(
        age_group_id=get_age_group_id(),
        location_id=location_id,
        year_id=year_id,
        sex_id=SEX.MALE + SEX.FEMALE + SEX.COMBINED,
        release_id=RELEASE_IDS.GBD_2021,
        forecasted_pop=True,
    )
    data = utilities.normalize_sex(
        data.drop("run_id", axis="columns").rename(columns={"population": "value"}),
        fill_value=None,
        cols_to_fill=utilities.DRAW_COLUMNS,
    )
    data = utilities.reshape(data, ["value"])
    data = utilities.scrub_gbd_conventions(data, location)
    data = utilities.split_interval(
        data, interval_column="age", split_column_prefix="age"
    )
    data = utilities.split_interval(
        data, interval_column="year", split_column_prefix="year"
    )
    return utilities.sort_hierarchical_data(data)

In [12]:
pop = get_population_future(location.title(), 2030).value.reindex(asfr.index)
pop

location  sex     age_start  age_end     year_start  year_end
Ethiopia  Female  0.000000   0.019178    2030        2031         34555.502815
                  0.019178   0.076712    2030        2031        102821.618998
                  0.076712   0.500000    2030        2031                  NaN
                  0.500000   1.000000    2030        2031                  NaN
                                                                     ...      
          Male    80.000000  85.000000   2030        2031        215000.475056
                  85.000000  90.000000   2030        2031        100478.541795
                  90.000000  95.000000   2030        2031         33156.550480
                  95.000000  125.000000  2030        2031          8245.199268
Name: value, Length: 50, dtype: float64

In [13]:
# Forecasted population does not have younger ages, but luckily none of these are WRA
assert (pop.index.get_level_values("age_end")[pop.isna()] < 10).all()
pop[pop.isna()]

location  sex     age_start  age_end  year_start  year_end
Ethiopia  Female  0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
          Male    0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
Name: value, dtype: float64

In [14]:
pop = pop.fillna(0)

In [15]:
n_births = (pop * asfr).sum()
n_births

3717119.066255957

In [16]:
sbr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.stillbirth_to_live_birth_ratio,
    "estimate",
    location.title(),
    years=2022,
).value
sbr

location  year_start  year_end  parameter  
Ethiopia  2022        2023      lower_value    0.017252
                                mean_value     0.017252
                                upper_value    0.017252
Name: value, dtype: float64

In [17]:
sbr = sbr[sbr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
sbr

location  year_start  year_end
Ethiopia  2022        2023        0.017252
Name: value, dtype: float64

In [18]:
sbr = sbr.values[0]

In [19]:
births_and_stillbirths = n_births + n_births * sbr
births_and_stillbirths / 1e6

3.7812475607071954

## Fertility (technically birth-and-stillbirth) disparities

In [20]:
if location == "india":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/IND/2019_2021/IND_DHS7_2019_2021_REP_FINAL_Y2022M05D11.PDF
        {  # Table 8.4 Perinatal mortality -- using "Number of pregnancies of 7 or more months' duration" as a proxy
            1: 56_979,
            2: 50_335,
            3: 45_189,
            4: 42_611,
            5: 36_290,
        }
    )
elif location == "nigeria":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/NGA/2018/NGA_DHS7_2018_REP_QUEST_Y2019M11D05.PDF
        {  # Table 8.4 Perinatal mortality
            1: 7_712,
            2: 7_886,
            3: 7_139,
            4: 6_328,
            5: 5_558,
        }
    )
elif location == "ethiopia":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/ETH/2016/ETH_DHS7_2016_REP_QUEST_Y2017M08D15.PDF
        {  # Table 8.4 Perinatal mortality
            1: 2_645,
            2: 2_516,
            3: 2_290,
            4: 2_018,
            5: 1_592,
        }
    )

dist_births_and_stillbirths_by_wealth.index.name = "wealth_quintile"

s_births = (
    n_births
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births

wealth_quintile
1    888868.992880
2    845517.726309
3    769568.995726
4    678161.673963
5    535001.677378
dtype: float64

In [21]:
s_births_and_stillbirths_by_wealth = (
    births_and_stillbirths
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births_and_stillbirths_by_wealth

wealth_quintile
1    904203.941603
2    860104.770160
3    782845.756624
4    689861.457147
5    544231.635173
dtype: float64

In [22]:
# http://ihmeuw.org/6jr2 -- extracted from GBD Foresight, count of NTD deaths in 2030 for under-1 year olds
# from GBD 2021
if location == "india":
    ntd_deaths = 4_273.37
elif location == "nigeria":
    ntd_deaths = 5_373.52
elif location == "ethiopia":
    ntd_deaths = 1_883.76


ntd_death_rate_per_birth = ntd_deaths / n_births
10_000 * ntd_death_rate_per_birth

5.067795694522652

In [23]:
# Assumed does not vary by wealth
champs_ntd_stillbirth_per_livebirth = 51 / (69 - 51)
ntd_stillbirths = ntd_deaths * champs_ntd_stillbirth_per_livebirth
10_000 * (
    ntd_deaths + ntd_stillbirths
) / n_births  # ntd rate, compare with 41 per 10,000 from Bhide et al https://pubmed.ncbi.nlm.nih.gov/23873811/

19.426550162336838

We could not find a good source for folate intake by wealth in India, or even a representative source for overall folate intake.  [This paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10755415/pdf/S1368980023002112a.pdf) has an overall number of 220 mcg/day for women, and since it is not too different from the values we found in Ethiopia and Nigeria, we are going to use it for now. (It also has standard deviation of 50, which we can use when we introduce heterogeneity)

In [24]:
if location == "india":
    folate_intake_by_wealth = pd.Series(
        {
            1: 220,
            2: 220,
            3: 220,
            4: 220,
            5: 220,  # NRV is 400 mcg/day
        }
    )
elif location == "nigeria":
    # Table 95 of NFCMS 2021 Report
    folate_intake_by_wealth = pd.Series(
        {
            1: 189,
            2: 198,
            3: 197,
            4: 203,
            5: 208,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?
elif location == "ethiopia":
    # Table 6 of https://cdn.nutrition.org/article/S2475-2991%2824%2901728-1/fulltext
    folate_intake_by_wealth = pd.Series(
        {
            1: 166,
            2: 152,
            3: 137,
            4: 350,
            5: 469,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?

folate_intake_by_wealth.index.name = "wealth_quintile"

In [25]:
if location == "india":
    s_dist_deaths_by_wealth = pd.Series(  # Table 7.9 on page 201 of the CNNS report has RBC folate deficiency rates;
        {  # it includes wealth stratification, but has a very low threshold for insufficiency
            1: 1,  # so I am assuming that most everyone is in the danger zone for low folate
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "nigeria":
    s_dist_deaths_by_wealth = pd.Series(  # assume same rate for all, for now;
        {  # can CHAMPS offer more detail?  Need to infer wealth somehow
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "ethiopia":
    s_dist_deaths_by_wealth = pd.Series(
        {  # supplementation studies don't make this easy, but here is a guess
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
    s_dist_deaths_by_wealth /= s_dist_deaths_by_wealth.mean()

s_dist_deaths_by_wealth.index.name = "wealth_quintile"
s_dist_deaths_by_wealth

wealth_quintile
1    1.0
2    1.0
3    1.0
4    1.0
5    1.0
dtype: float64

In [26]:
s_ntd_death_rate_per_birth = ntd_deaths / n_births * s_dist_deaths_by_wealth
10_000 * s_ntd_death_rate_per_birth

wealth_quintile
1    5.067796
2    5.067796
3    5.067796
4    5.067796
5    5.067796
dtype: float64

In [27]:
s_ntd_death_count = s_ntd_death_rate_per_birth * s_births
s_ntd_death_count

wealth_quintile
1    450.460646
2    428.491109
3    390.001844
4    343.678481
5    271.127920
dtype: float64

In [28]:
assert np.isclose(s_ntd_death_count.sum(), ntd_deaths)

In [29]:
s_ntd_stillbirth_count = s_ntd_death_count * champs_ntd_stillbirth_per_livebirth
s_ntd_stillbirth_count

wealth_quintile
1    1276.305162
2    1214.058143
3    1105.005226
4     973.755697
5     768.195773
dtype: float64

In [30]:
s_ntd_death_or_stillbirth_count = s_ntd_death_count + s_ntd_stillbirth_count
s_ntd_death_or_stillbirth_count

wealth_quintile
1    1726.765808
2    1642.549252
3    1495.007070
4    1317.434178
5    1039.323692
dtype: float64

In [31]:
def backcalc_rbc(ntd_risk, method):
    """
    ln (odds of NTD risk) = 1.6563 − 1.2193 × ln (RBC) (Daly et al, 1995)
    ln (odds of NTD risk) = 4.57 − 1.70 × ln (RBC) (Crider et al, 2014)
    """

    odds = ntd_risk / (1 - ntd_risk)
    ln_odds = np.log(odds)
    if method == "daly":
        neg_ln_rbc = (ln_odds - 1.6563) / 1.2193
    elif method == "crider":
        neg_ln_rbc = (ln_odds - 4.57) / 1.70
    rbc = np.exp(-neg_ln_rbc)
    return rbc


backcalc_rbc(s_ntd_death_or_stillbirth_count / s_births, "daly")

wealth_quintile
1    650.36406
2    650.36406
3    650.36406
4    650.36406
5    650.36406
dtype: float64

In [32]:
backcalc_rbc(
    s_ntd_death_or_stillbirth_count / s_births, "crider"
)  # compare with CNNS, https://www.unicef.org/india/media/2646/file/CNNS-report.pdf in Table 7.9

wealth_quintile
1    578.163072
2    578.163072
3    578.163072
4    578.163072
5    578.163072
dtype: float64

In [33]:
pop

location  sex     age_start  age_end     year_start  year_end
Ethiopia  Female  0.000000   0.019178    2030        2031         34555.502815
                  0.019178   0.076712    2030        2031        102821.618998
                  0.076712   0.500000    2030        2031             0.000000
                  0.500000   1.000000    2030        2031             0.000000
                                                                     ...      
          Male    80.000000  85.000000   2030        2031        215000.475056
                  85.000000  90.000000   2030        2031        100478.541795
                  90.000000  95.000000   2030        2031         33156.550480
                  95.000000  125.000000  2030        2031          8245.199268
Name: value, Length: 50, dtype: float64

In [34]:
s_pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
s_pop = s_pop.set_index([c for c in s_pop.columns if c != "value"])
s_pop

value
sex    age_start age_end    pregnant     wealth_quintile             
Female 0.0       0.019178   not_pregnant 1                7729.330504
                                         2                7026.085890
                                         3                6642.401601
                                         4                5956.884779
...                                                               ...
Male   95.0      125.000000 not_pregnant 2                 690.099645
                                         3                 716.289850
                                         4                 757.897451
                                         5                 664.219250

[285 rows x 1 columns]

In [35]:
# WRA only
s_pop = s_pop[
    (s_pop.index.get_level_values("sex") == "Female")
    & (s_pop.index.get_level_values("age_start") >= 15)
    & (s_pop.index.get_level_values("age_end") <= 50)
].copy()

In [36]:
s_daily_vehicle = pd.read_csv(
    f"../0100_data_prep/results/{vehicle}/vehicle_consumption/amount/mean/{location}.csv"
)
assert (s_daily_vehicle.vehicle_name == vehicle).all()
s_daily_vehicle = s_daily_vehicle.drop(columns=["vehicle_name"])
s_daily_vehicle = s_daily_vehicle.set_index(
    [c for c in s_daily_vehicle.columns if c != "value"]
).value
s_daily_vehicle

wealth_quintile  sex     age_start  age_end
1                Female  15         50         7.1
2                Female  15         50         7.2
3                Female  15         50         7.7
4                Female  15         50         7.2
5                Female  15         50         6.5
Name: value, dtype: float64

In [37]:
from lsff_utils import data_processing

In [38]:
s_daily_vehicle = (
    s_pop.value
    * data_processing.reindex_series_onto_df_by_age_groups(s_pop, s_daily_vehicle)
).groupby("wealth_quintile").sum() / s_pop.value.groupby("wealth_quintile").sum()
s_daily_vehicle

wealth_quintile
1    7.1
2    7.2
3    7.7
4    7.2
5    6.5
Name: value, dtype: float64

In [39]:
any_consumers = pd.read_csv(
    f"../0100_data_prep/results/{vehicle}/vehicle_consumption/any/{location}.csv"
)
assert (any_consumers.vehicle_name == vehicle).all()
any_consumers = any_consumers.drop(columns=["vehicle_name"])
any_consumers = any_consumers.set_index(
    [c for c in any_consumers.columns if c != "value"]
).value
any_consumers

wealth_quintile  sex     age_start  age_end
1                Female  15         50         0.97
2                Female  15         50         0.97
3                Female  15         50         0.97
4                Female  15         50         0.97
5                Female  15         50         0.97
Name: value, dtype: float64

In [40]:
any_consumers = (
    s_pop.value
    * data_processing.reindex_series_onto_df_by_age_groups(s_pop, any_consumers)
).groupby("wealth_quintile").sum() / s_pop.value.groupby("wealth_quintile").sum()
any_consumers

wealth_quintile
1    0.97
2    0.97
3    0.97
4    0.97
5    0.97
Name: value, dtype: float64

In [41]:
s_daily_vehicle_among_consumers = s_daily_vehicle / any_consumers
s_daily_vehicle_among_consumers

wealth_quintile
1    7.319588
2    7.422680
3    7.938144
4    7.422680
5    6.701031
Name: value, dtype: float64

In [42]:
baseline_concentration_mcg_per_gram = pd.read_csv(
    f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/concentration/{location}.csv"
)
assert (baseline_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert baseline_concentration_mcg_per_gram.value.nunique() == 1
baseline_concentration_mcg_per_gram = baseline_concentration_mcg_per_gram.value.iloc[0]
baseline_concentration_mcg_per_gram

0.0

In [43]:
intervention_concentration_mcg_per_gram = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/concentration/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (intervention_concentration_mcg_per_gram.vehicle_name == vehicle).all()
intervention_concentration_mcg_per_gram = (
    intervention_concentration_mcg_per_gram.drop(columns=["vehicle_name"])
    .set_index("scenario")
    .value
)
intervention_concentration_mcg_per_gram

scenario
intervention_25_nrv     14.084507
intervention_100_nrv    56.338028
Name: value, dtype: float64

In [44]:
eff_fort_baseline_path = f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/effective_coverage/{location}.csv"

df_eff_fort_baseline = pd.read_csv(eff_fort_baseline_path)
assert (df_eff_fort_baseline.vehicle_name == vehicle).all()

In [45]:
if location == "india" and vehicle == "rice":
    # Confusingly, our baseline scenario (our best guess about the present)
    # is *not* a good guess about 2019-2020 (which is when our baseline folate estimate is from),
    # because this program has rolled out entirely since then:
    # In the phase-I of the roll out, the fortified rice was introduced in the social welfare schemes such as Integrated Child Development Scheme (ICDS)
    # and Pradhan Mantri Poshan Shakti Nirman (PM POSHAN, earlier known as the National Program of Mid-Day Meal in Schools)
    # throughout India during 2021–22 [18].
    # Phase-II has covered aspirational and high burden districts for anemia (total 291 districts) under Public Distribution System (PDS) and other welfare schemes,
    # in addition to Phase-I districts, by March 2023 [18].
    # All the remaining districts in India will be covered in Phase III by March 2024 [19].
    # ~ https://pmc.ncbi.nlm.nih.gov/articles/PMC11305529/
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline.assign(value=0)
else:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline

In [46]:
df_eff_fort_intervention = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (df_eff_fort_intervention.vehicle_name == vehicle).all()
df_eff_fort_intervention = df_eff_fort_intervention.drop(columns=["vehicle_name"])
df_eff_fort_intervention

,wealth_quintile,sex,value,scenario
0,1,Female,0.6984,intervention_25_nrv
1,1,Male,0.6984,intervention_25_nrv
2,2,Female,0.6984,intervention_25_nrv
3,2,Male,0.6984,intervention_25_nrv
...,...,...,...,...
6,4,Female,0.6984,intervention_100_nrv
7,4,Male,0.6984,intervention_100_nrv
8,5,Female,0.6984,intervention_100_nrv
9,5,Male,0.6984,intervention_100_nrv


In [47]:
# NOTE: Using DHS definition of WRA
population = (
    pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
    .groupby(["sex", "age_start", "age_end", "wealth_quintile"])
    .value.sum()
    .reset_index()
)
population = population[
    (population.sex == "Female")
    & (population.age_start >= 15)
    & (population.age_end <= 50)
]
population

,sex,age_start,age_end,wealth_quintile,value
40,Female,15.0,20.0,1,9.914727e+05
41,Female,15.0,20.0,2,1.121924e+06
42,Female,15.0,20.0,3,1.134915e+06
43,Female,15.0,20.0,4,1.212617e+06
...,...,...,...,...,...
71,Female,45.0,50.0,2,3.254899e+05
72,Female,45.0,50.0,3,3.622846e+05
73,Female,45.0,50.0,4,3.746773e+05
74,Female,45.0,50.0,5,4.350470e+05


In [48]:
if "sex" in df_eff_fort_baseline_2019_2020.columns:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline_2019_2020[(df_eff_fort_baseline_2019_2020.sex == "Female")]

if "age_start" in df_eff_fort_baseline_2019_2020.columns:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline_2019_2020[
        (df_eff_fort_baseline_2019_2020.age_start >= 15) & (df_eff_fort_baseline_2019_2020.age_end <= 50)
    ]

In [49]:
if "sex" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.sex == "Female")
    ]

if "age_start" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.age_start >= 15)
        & (df_eff_fort_intervention.age_end <= 50)
    ]

In [50]:
def aggregate_using_population(effective_fort):
    merge_cols = [
        c
        for c in ["sex", "wealth_quintile", "age_start", "age_end"]
        if c in effective_fort.columns
    ]
    merged = effective_fort.merge(
        population.reset_index(),
        on=[c for c in merge_cols if "age" not in c],
        suffixes=("_fort", "_pop"),
    )
    assert ("age_start" in merge_cols) == ("age_end" in merge_cols)
    if "age_start" in merge_cols:
        merged = merged[
            (merged.age_start_pop >= merged.age_start_fort)
            & (merged.age_end_pop <= merged.age_end_fort)
        ]
    print(merged)
    assert len(merged) == len(population) * (
        1
        if "scenario" not in effective_fort.columns
        else effective_fort.scenario.nunique()
    )
    group_cols = [c for c in ["scenario", "wealth_quintile"] if c in merged]
    return merged.groupby(group_cols).apply(
        lambda df: (df.value_fort * df.value_pop).sum() / df.value_pop.sum()
    )

In [51]:
df_eff_fort_baseline_2019_2020 = aggregate_using_population(df_eff_fort_baseline_2019_2020)
df_eff_fort_baseline_2019_2020

   vehicle_name  wealth_quintile  value_fort  index     sex  age_start  \
0          salt                1         0.0     40  Female       15.0   
1          salt                1         0.0     45  Female       20.0   
2          salt                1         0.0     50  Female       25.0   
3          salt                1         0.0     55  Female       30.0   
..          ...              ...         ...    ...     ...        ...   
31         salt                5         0.0     59  Female       30.0   
32         salt                5         0.0     64  Female       35.0   
33         salt                5         0.0     69  Female       40.0   
34         salt                5         0.0     74  Female       45.0   

    age_end      value_pop  
0      20.0  991472.669353  
1      25.0  914441.525853  
2      30.0  763913.783189  
3      35.0  719362.052894  
..      ...            ...  
31     35.0  848301.959351  
32     40.0  725030.396423  
33     45.0  595266.794557 

wealth_quintile
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
dtype: float64

In [52]:
df_eff_fort_baseline = aggregate_using_population(df_eff_fort_baseline)
df_eff_fort_baseline

   vehicle_name  wealth_quintile  value_fort  index     sex  age_start  \
0          salt                1         0.0     40  Female       15.0   
1          salt                1         0.0     45  Female       20.0   
2          salt                1         0.0     50  Female       25.0   
3          salt                1         0.0     55  Female       30.0   
..          ...              ...         ...    ...     ...        ...   
31         salt                5         0.0     59  Female       30.0   
32         salt                5         0.0     64  Female       35.0   
33         salt                5         0.0     69  Female       40.0   
34         salt                5         0.0     74  Female       45.0   

    age_end      value_pop  
0      20.0  991472.669353  
1      25.0  914441.525853  
2      30.0  763913.783189  
3      35.0  719362.052894  
..      ...            ...  
31     35.0  848301.959351  
32     40.0  725030.396423  
33     45.0  595266.794557 

wealth_quintile
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
dtype: float64

In [53]:
df_eff_fort_intervention = aggregate_using_population(df_eff_fort_intervention)
df_eff_fort_intervention

    wealth_quintile     sex  value_fort              scenario  index  \
0                 1  Female      0.6984   intervention_25_nrv     40   
1                 1  Female      0.6984   intervention_25_nrv     45   
2                 1  Female      0.6984   intervention_25_nrv     50   
3                 1  Female      0.6984   intervention_25_nrv     55   
..              ...     ...         ...                   ...    ...   
66                5  Female      0.6984  intervention_100_nrv     59   
67                5  Female      0.6984  intervention_100_nrv     64   
68                5  Female      0.6984  intervention_100_nrv     69   
69                5  Female      0.6984  intervention_100_nrv     74   

    age_start  age_end      value_pop  
0        15.0     20.0  991472.669353  
1        20.0     25.0  914441.525853  
2        25.0     30.0  763913.783189  
3        30.0     35.0  719362.052894  
..        ...      ...            ...  
66       30.0     35.0  848301.959351  

scenario              wealth_quintile
intervention_100_nrv  1                  0.6984
                      2                  0.6984
                      3                  0.6984
                      4                  0.6984
                                          ...  
intervention_25_nrv   2                  0.6984
                      3                  0.6984
                      4                  0.6984
                      5                  0.6984
Length: 10, dtype: float64

In [54]:
RBC_baseline = backcalc_rbc(s_ntd_death_or_stillbirth_count / s_births, "crider")
RBC_baseline

wealth_quintile
1    578.163072
2    578.163072
3    578.163072
4    578.163072
5    578.163072
dtype: float64

In [55]:
# Fortification folate needs to be converted into dietary folate equivalents (DFEs)
# for use with our effect size.
# https://www.jandonline.org/article/S0002-8223(00)00027-4/pdf
fortification_mcg_to_dfe = 1.7

In [56]:
# Delete fortification effect baked into our baseline folate estimate.
# In non-India locations, this is going to be zero.
# For India, our current source for baseline folate is very rough,
# but it does appear to be from before the fortification program (2019-2020).
s_zero_folate = folate_intake_by_wealth - (
    df_eff_fort_baseline_2019_2020
    * s_daily_vehicle_among_consumers
    * baseline_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)

In [57]:
s_baseline_folate = s_zero_folate + (
    df_eff_fort_baseline
    * s_daily_vehicle_among_consumers
    * baseline_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)
s_baseline_folate

wealth_quintile
1    166.0
2    152.0
3    137.0
4    350.0
5    469.0
dtype: float64

In [58]:
s_intervention_folate = s_zero_folate + (
    df_eff_fort_intervention
    * s_daily_vehicle_among_consumers
    * intervention_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)
s_intervention_folate

scenario              wealth_quintile
intervention_100_nrv  1                  655.600000
                      2                  648.495775
                      3                  667.974648
                      4                  846.495775
                                            ...    
intervention_25_nrv   2                  276.123944
                      3                  269.743662
                      4                  474.123944
                      5                  581.056338
Length: 10, dtype: float64

In [59]:
zero_folate_pct_decrease = (folate_intake_by_wealth - s_zero_folate) / folate_intake_by_wealth

In [60]:
baseline_folate_pct_increase_from_zero = (
    s_baseline_folate - s_zero_folate
) / folate_intake_by_wealth
baseline_folate_pct_increase_from_zero

wealth_quintile
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
dtype: float64

In [61]:
intevention_folate_pct_increase_from_zero = (
    s_intervention_folate - s_zero_folate
) / folate_intake_by_wealth
intevention_folate_pct_increase_from_zero

scenario              wealth_quintile
intervention_100_nrv  1                  2.949398
                      2                  3.266420
                      3                  3.875727
                      4                  1.418559
                                           ...   
intervention_25_nrv   2                  0.816605
                      3                  0.968932
                      4                  0.354640
                      5                  0.238926
Length: 10, dtype: float64

In [62]:
RBC_zero = RBC_baseline / (1 + ((6 / 10) * zero_folate_pct_decrease))
RBC_zero

wealth_quintile
1    578.163072
2    578.163072
3    578.163072
4    578.163072
5    578.163072
dtype: float64

In [63]:
RBC_baseline = RBC_zero * (
    1 + ((6 / 10) * baseline_folate_pct_increase_from_zero)
)
RBC_baseline

wealth_quintile
1    578.163072
2    578.163072
3    578.163072
4    578.163072
5    578.163072
dtype: float64

In [64]:
RBC_intervention = RBC_zero * (
    1 + ((6 / 10) * intevention_folate_pct_increase_from_zero)
)
RBC_intervention

scenario              wealth_quintile
intervention_100_nrv  1                  1601.302734
                      2                  1711.276975
                      3                  1922.644532
                      4                  1070.258253
                                            ...     
intervention_25_nrv   2                   861.441548
                      3                   914.283437
                      4                   701.186867
                      5                   661.046018
Length: 10, dtype: float64

In [65]:
def calc_ntd_pr(df, method):
    ln_rbc = np.log(df)
    if method == "daly":
        ln_odds = 1.6563 - 1.2193 * ln_rbc
    elif method == "crider":
        ln_odds = 4.57 - 1.70 * ln_rbc
    p = np.exp(ln_odds)  # TODO: better transformation
    return p

In [66]:
# NOTE: All rates here are per birth!
s_ntd_death_or_stillbirth_rate_zero = calc_ntd_pr(RBC_zero, "crider")
10_000 * s_ntd_death_or_stillbirth_rate_zero

wealth_quintile
1    19.464363
2    19.464363
3    19.464363
4    19.464363
5    19.464363
dtype: float64

In [67]:
s_ntd_death_or_stillbirth_rate_baseline = calc_ntd_pr(RBC_baseline, "crider")
10_000 * s_ntd_death_or_stillbirth_rate_baseline

wealth_quintile
1    19.464363
2    19.464363
3    19.464363
4    19.464363
5    19.464363
dtype: float64

In [68]:
s_ntd_death_or_stillbirth_rate_intervention = calc_ntd_pr(RBC_intervention, "crider")
10_000 * s_ntd_death_or_stillbirth_rate_intervention

scenario              wealth_quintile
intervention_100_nrv  1                   3.444458
                      2                   3.076672
                      3                   2.524047
                      4                   6.832745
                                           ...    
intervention_25_nrv   2                   9.881933
                      3                   8.930758
                      4                  14.021948
                      5                  15.500007
Length: 10, dtype: float64

In [69]:
s_ntd_death_or_stillbirth_count_zero = s_ntd_death_or_stillbirth_rate_zero * s_births
s_ntd_death_or_stillbirth_count_zero

wealth_quintile
1    1730.126847
2    1645.746370
3    1497.917006
4    1319.998479
5    1041.346670
dtype: float64

In [70]:
s_ntd_death_or_stillbirth_count_baseline = (
    s_ntd_death_or_stillbirth_rate_baseline * s_births
)
s_ntd_death_or_stillbirth_count_baseline

wealth_quintile
1    1730.126847
2    1645.746370
3    1497.917006
4    1319.998479
5    1041.346670
dtype: float64

In [71]:
s_ntd_death_or_stillbirth_count_intervention = (
    s_ntd_death_or_stillbirth_rate_intervention * s_births
)
s_ntd_death_or_stillbirth_count_intervention

scenario              wealth_quintile
intervention_100_nrv  1                  306.167214
                      2                  260.138108
                      3                  194.242854
                      4                  463.370551
                                            ...    
intervention_25_nrv   2                  835.534939
                      3                  687.283466
                      4                  950.914807
                      5                  829.252991
Length: 10, dtype: float64

In [72]:
ntd_cases_by_scenario = pd.concat(
    [
        s_ntd_death_or_stillbirth_count_zero.rename("value")
        .to_frame()
        .assign(entity="ntd", scenario="zero")
        .set_index(["entity", "scenario"], append=True)
        .value,
        s_ntd_death_or_stillbirth_count_baseline.rename("value")
        .to_frame()
        .assign(entity="ntd", scenario="baseline")
        .set_index(["entity", "scenario"], append=True)
        .value,
        *[
            s_ntd_death_or_stillbirth_count_intervention.loc[intervention_scenario]
            .rename("value")
            .to_frame()
            .assign(entity="ntd", scenario=intervention_scenario)
            .set_index(["entity", "scenario"], append=True)
            .value
            for intervention_scenario in intervention_scenarios
        ],
    ]
)
ntd_cases_by_scenario

wealth_quintile  entity  scenario            
1                ntd     zero                    1730.126847
2                ntd     zero                    1645.746370
3                ntd     zero                    1497.917006
4                ntd     zero                    1319.998479
                                                    ...     
2                ntd     intervention_100_nrv     260.138108
3                ntd     intervention_100_nrv     194.242854
4                ntd     intervention_100_nrv     463.370551
5                ntd     intervention_100_nrv     481.901378
Name: value, Length: 20, dtype: float64

In [73]:
(
    ntd_cases_by_scenario[ntd_cases_by_scenario.index.get_level_values('scenario') == 'baseline'].droplevel('scenario') -
    ntd_cases_by_scenario[ntd_cases_by_scenario.index.get_level_values('scenario') == 'intervention'].droplevel('scenario')
)

wealth_quintile  entity
1                ntd      NaN
2                ntd      NaN
3                ntd      NaN
4                ntd      NaN
5                ntd      NaN
Name: value, dtype: float64

In [74]:
(
    ntd_cases_by_scenario[ntd_cases_by_scenario.index.get_level_values('scenario') == 'zero'].droplevel('scenario') -
    ntd_cases_by_scenario[ntd_cases_by_scenario.index.get_level_values('scenario') == 'baseline'].droplevel('scenario')
)

wealth_quintile  entity
1                ntd       0.0
2                ntd       0.0
3                ntd       0.0
4                ntd       0.0
5                ntd       0.0
Name: value, dtype: float64

In [75]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)

In [76]:
# For calculating YLLs
tmrle = vivarium_inputs.get_theoretical_minimum_risk_life_expectancy()
tmrle

,,value
age_start,age_end,
0.00,0.01,89.958040
0.01,0.02,89.975474
0.02,0.03,89.990990
0.03,0.04,89.985077
...,...,...
109.97,109.98,4.509941
109.98,109.99,4.504631
109.99,110.00,4.499321
110.00,125.00,4.494011


In [77]:
# NOTE: Treating stillbirths as a death!
yll_per_ntd = float(tmrle.iloc[0])
yll_per_ntd

89.95803974533831

In [78]:
ylls_by_scenario = ntd_cases_by_scenario * yll_per_ntd
ylls_by_scenario

wealth_quintile  entity  scenario            
1                ntd     zero                    155638.819702
2                ntd     zero                    148048.117342
3                ntd     zero                    134749.677549
4                ntd     zero                    118744.475674
                                                     ...      
2                ntd     intervention_100_nrv     23401.514260
3                ntd     intervention_100_nrv     17473.706345
4                ntd     intervention_100_nrv     41683.906456
5                ntd     intervention_100_nrv     43350.903288
Name: value, Length: 20, dtype: float64

In [79]:
path = f"./results/{location}/{vehicle}/ylls_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylls_by_scenario.to_csv(path)